In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Cities 2022 

In [6]:
wci_2022 = pd.read_csv("../data/city_safety/world-crime-index.csv")

In [7]:
wci_2022.head()

,Rank,City,Crime Index,Safety Index
0,1,"Caracas, Venezuela",83.98,16.02
1,2,"Pretoria, South Africa",81.98,18.02
2,3,"Celaya, Mexico",81.80,18.20
3,4,"San Pedro Sula, Honduras",80.87,19.13
4,5,"Port Moresby, Papua New Guinea",80.71,19.29


In [8]:
# Normalize column names (lowercase, underscores)
wci_2022.columns = (
    wci_2022.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [9]:
wci_2022.head()

,rank,city,crime_index,safety_index
0,1,"Caracas, Venezuela",83.98,16.02
1,2,"Pretoria, South Africa",81.98,18.02
2,3,"Celaya, Mexico",81.80,18.20
3,4,"San Pedro Sula, Honduras",80.87,19.13
4,5,"Port Moresby, Papua New Guinea",80.71,19.29


In [10]:
# Split city into city + country on comma
city_country = wci_2022['city'].str.rsplit(',', n=1, expand=True)

wci_2022['city'] = city_country[0].str.strip()
wci_2022['country'] = city_country[1].str.strip()

In [11]:
wci_2022.head()

,rank,city,crime_index,safety_index,country
0,1,Caracas,83.98,16.02,Venezuela
1,2,Pretoria,81.98,18.02,South Africa
2,3,Celaya,81.80,18.20,Mexico
3,4,San Pedro Sula,80.87,19.13,Honduras
4,5,Port Moresby,80.71,19.29,Papua New Guinea


In [12]:
# Add normalized versions for joins later
wci_2022['city_norm'] = wci_2022['city'].str.strip().str.lower()
wci_2022['country_norm'] = wci_2022['country'].str.strip().str.lower()

In [13]:
# Check for missing values
wci_2022.isna().sum()

rank            0
city            0
crime_index     0
safety_index    0
country         0
city_norm       0
country_norm    0
dtype: int64

In [14]:
wci_2022.isna().mean()

rank            0.0
city            0.0
crime_index     0.0
safety_index    0.0
country         0.0
city_norm       0.0
country_norm    0.0
dtype: float64

In [15]:
# Save cleaned dataframe back into data/city_safety
out_path = "../data/city_safety/world_crime_index_2022_clean.csv"
wci_2022.to_csv(out_path, index=False)

In [16]:
pd.read_csv("../data/city_safety/world_crime_index_2022_clean.csv").head()

,rank,city,crime_index,safety_index,country,city_norm,country_norm
0,1,Caracas,83.98,16.02,Venezuela,caracas,venezuela
1,2,Pretoria,81.98,18.02,South Africa,pretoria,south africa
2,3,Celaya,81.80,18.20,Mexico,celaya,mexico
3,4,San Pedro Sula,80.87,19.13,Honduras,san pedro sula,honduras
4,5,Port Moresby,80.71,19.29,Papua New Guinea,port moresby,papua new guinea


# Countries 2024

In [17]:
# Load global country-level crime/safety data
global_path = "../data/crime-rate-by-country-2024.csv"  # from https://www.kaggle.com/datasets/shahriarkabir/crime-rate-by-country-2024?resource=download
global_df = pd.read_csv(global_path)

global_df.head()

,country,crimeRateByCountry_crimeIndex,CrimeRate_OverallCriminalityScoreGOCI,CrimeRate_CriminalMarketsScore,CrimeRate_CriminalActorsScore,CrimeRate_ResilienceScore,CrimeRateSafetyIndex
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [18]:
print(global_df.columns)

Index(['country', 'crimeRateByCountry_crimeIndex',
       'CrimeRate_OverallCriminalityScoreGOCI',
       'CrimeRate_CriminalMarketsScore', 'CrimeRate_CriminalActorsScore',
       'CrimeRate_ResilienceScore', 'CrimeRateSafetyIndex'],
      dtype='object')


In [19]:
# Standardize column names
global_df.columns = (
    global_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Rename to a cleaner schema

global_df = global_df.rename(columns={
    "country_name": "country",
    "crimeratebycountry_crimeindex": "crime_index",
    "crimerate_overallcriminalityscoregoci": "overall_criminality_score",
    "crimerate_criminalmarketsscore": "criminal_markets_score",
    "crimerate_criminalactorsscore": "criminal_actors_score",
    "crimerate_resiliencescore": "resilience_score",
    "crimeratesafetyindex": "safety_index"
})

global_df.head()


,country,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [20]:
# Missing counts per column
print(global_df.isna().sum())

# Fraction missing per column
print(global_df.isna().mean())

# Which countries have any missing numeric values
num_cols = [
    'crime_index',
    'overall_criminality_score',
    'criminal_markets_score',
    'criminal_actors_score',
    'resilience_score',
    'safety_index',
]

global_df['num_missing'] = global_df[num_cols].isna().sum(axis=1)

countries_with_missing = global_df[global_df['num_missing'] > 0][
    ['country', 'num_missing'] + num_cols
]

countries_with_missing.head(50)

country                       0
crime_index                  57
overall_criminality_score     6
criminal_markets_score        6
criminal_actors_score         6
resilience_score              6
safety_index                 58
dtype: int64
country                      0.000000
crime_index                  0.287879
overall_criminality_score    0.030303
criminal_markets_score       0.030303
criminal_actors_score        0.030303
resilience_score             0.030303
safety_index                 0.292929
dtype: float64


,country,num_missing,crime_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score,safety_index
14,DR Congo,2,NaN,7.35,6.20,8.5,2.38,NaN
19,Thailand,2,NaN,6.18,6.77,5.6,4.79,NaN
27,Colombia,2,NaN,7.75,7.30,8.2,5.63,NaN
36,Yemen,2,NaN,6.57,5.63,7.5,1.75,NaN
37,Canada,2,NaN,3.88,3.87,3.9,7.21,NaN
48,Madagascar,2,NaN,5.58,5.27,5.9,3.33,NaN
51,Cameroon,2,NaN,6.27,6.23,6.3,3.17,NaN
53,Niger,2,NaN,5.70,5.70,5.7,3.46,NaN
57,Mali,2,NaN,5.93,6.47,5.4,2.38,NaN
59,Taiwan,4,16.1,NaN,NaN,NaN,NaN,83.9


In [ ]:
#out_path = "../data/country_safety/country_crime_safety_2024_clean.csv"
#global_df.to_csv(out_path, index=False)

# need to clean or decide on fill values before saving  

# Countries 2020 - this dataset came along with many other csvs that could be used as features

In [22]:
#https://www.kaggle.com/datasets/dumbgeek/countries-dataset-2020?select=Crime+index+by+countries+2020.csv
global_df_1 = pd.read_csv("../data/Crime_index_by_countries_2020.csv")
global_df_1.head()

,Country,Crime Index,Safety Index
0,Venezuela,84.49,15.51
1,Papua New Guinea,81.93,18.07
2,South Africa,77.49,22.51
3,Afghanistan,76.23,23.77
4,Honduras,76.11,23.89


In [23]:
print(global_df_1.columns)

Index(['Country', 'Crime Index', 'Safety Index'], dtype='object')


In [24]:
# Normalize names
global_df_1.columns = (
    global_df_1.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

global_df_1.head()

,country,crime_index,safety_index
0,Venezuela,84.49,15.51
1,Papua New Guinea,81.93,18.07
2,South Africa,77.49,22.51
3,Afghanistan,76.23,23.77
4,Honduras,76.11,23.89


In [25]:
# Clean country text and add normalized key for merges
global_df_1['country'] = global_df_1['country'].astype(str).str.strip()
global_df_1['country_norm'] = global_df_1['country'].str.lower()

# Ensure numeric dtypes (coerce any weird strings to NaN)
num_cols = ['crime_index', 'safety_index']

for c in num_cols:
    global_df_1[c] = pd.to_numeric(global_df_1[c], errors='coerce')

In [26]:
# Duplicated countries (after normalization)
dupes = global_df_1[global_df_1['country_norm'].duplicated(keep=False)]\
        .sort_values('country_norm')
print(dupes.head(20))

# Missing counts
print(global_df_1[num_cols].isna().sum())
print(global_df_1[num_cols].isna().mean())

Empty DataFrame
Columns: [country, crime_index, safety_index, country_norm]
Index: []
crime_index     0
safety_index    0
dtype: int64
crime_index     0.0
safety_index    0.0
dtype: float64


In [28]:
global_df_1 = global_df_1.rename(columns={
    'crime_index': 'crime_index_2020',
    'safety_index': 'safety_index_2020'
})

out_path = "../data/country_safety/country_crime_safety_2020_clean.csv"
global_df_1.to_csv(out_path, index=False)

In [38]:
pd.read_csv("../data/country_safety/country_crime_safety_2020_clean.csv").head()

,country,crime_index_2020,safety_index_2020,country_norm
0,Venezuela,84.49,15.51,venezuela
1,Papua New Guinea,81.93,18.07,papua new guinea
2,South Africa,77.49,22.51,south africa
3,Afghanistan,76.23,23.77,afghanistan
4,Honduras,76.11,23.89,honduras


# Countries 2023 

In [31]:
#https://www.kaggle.com/datasets/zabihullah18/crime-rate-by-country-2023
global_df_2 = pd.read_csv("../data/crime_index_by_country_2023.csv")
global_df_2.head()

,rank,country,crimeIndex,pop2023
0,1,Venezuela,83.76,28838499.0
1,2,Papua New Guinea,80.79,10329931.0
2,3,South Africa,76.86,60414495.0
3,4,Afghanistan,76.31,42239854.0
4,5,Honduras,74.54,10593798.0


In [32]:
global_df_2.columns = (
    global_df_2.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

global_df_2.head()

,rank,country,crimeindex,pop2023
0,1,Venezuela,83.76,28838499.0
1,2,Papua New Guinea,80.79,10329931.0
2,3,South Africa,76.86,60414495.0
3,4,Afghanistan,76.31,42239854.0
4,5,Honduras,74.54,10593798.0


In [33]:
global_df_2['country'] = global_df_2['country'].astype(str).str.strip()
global_df_2['country_norm'] = global_df_2['country'].str.lower()

In [34]:
num_cols = ['crimeindex', 'pop2023']

for c in num_cols:
    global_df_2[c] = pd.to_numeric(global_df_2[c], errors='coerce')

In [35]:
# Duplicated countries
dupes_2023 = global_df_2[global_df_2['country_norm'].duplicated(keep=False)]\
               .sort_values('country_norm')
print(dupes_2023.head(20))

# Missing counts and fractions
print(global_df_2[num_cols].isna().sum())
print(global_df_2[num_cols].isna().mean())

Empty DataFrame
Columns: [rank, country, crimeindex, pop2023, country_norm]
Index: []
crimeindex    0
pop2023       0
dtype: int64
crimeindex    0.0
pop2023       0.0
dtype: float64


In [36]:
global_df_2 = global_df_2.rename(columns={
    'crimeindex': 'crime_index_2023',
    'pop2023': 'population_2023',
})

out_path_2023 = "../data/country_safety/country_crime_pop_2023_clean.csv"
global_df_2.to_csv(out_path_2023, index=False)

In [40]:
pd.read_csv("../data/country_safety/country_crime_pop_2023_clean.csv").head()

,rank,country,crime_index_2023,population_2023,country_norm
0,1,Venezuela,83.76,28838499.0,venezuela
1,2,Papua New Guinea,80.79,10329931.0,papua new guinea
2,3,South Africa,76.86,60414495.0,south africa
3,4,Afghanistan,76.31,42239854.0,afghanistan
4,5,Honduras,74.54,10593798.0,honduras


# Quality of life dataset with saftey index - there are a bunch of these to look into that are very crediable adn complete

In [47]:
#https://www.kaggle.com/datasets/ahmedmohamed2003/quality-of-life-for-each-country
qol_1 = pd.read_csv("../data/Quality_of_Life_1.csv")
qol_1.head()

,country,Purchasing Power Value,Purchasing Power Category,Safety Value,Safety Category,Health Care Value,Health Care Category,Climate Value,Climate Category,Cost of Living Value,Cost of Living Category,Property Price to Income Value,Property Price to Income Category,Traffic Commute Time Value,Traffic Commute Time Category,Pollution Value,Pollution Category,Quality of Life Value,Quality of Life Category
0,Afghanistan,32.15,'Very Low',25.33,'Low',24.24,'Low',0.00,None,21.08,'Very Low',7.8,'Low',56.17,'Very High',84.44,'Very High',0.0,None
1,Aland Islands,125.01,'Very High',71.81,'High',79.72,'High',0.00,None,53.44,'Low',5.33,'Low',19.05,'Very Low',18.05,'Very Low',0.0,None
2,Albania,42.82,'Low',55.52,'Moderate',48.21,'Moderate',86.43,'Very High',40.85,'Low',14.88,'High',36.74,'Moderate',77.25,'High',': 104.16','Low'
3,Alderney,0.00,None,83.79,'Very High',100.00,'Very High',0.00,None,0.00,None,0.0,None,5.00,'Very Low',1.72,'Very Low',0.0,None
4,Algeria,27.60,'Very Low',47.54,'Moderate',54.43,'Moderate',94.82,'Very High',25.31,'Very Low',21.7,'Very High',45.09,'High',63.87,'High',': 98.83','Very Low'


In [48]:
qol_1.columns

Index(['country', 'Purchasing Power Value', 'Purchasing Power Category',
       'Safety Value', 'Safety Category', 'Health Care Value',
       'Health Care Category', 'Climate Value', 'Climate Category',
       'Cost of Living Value', 'Cost of Living Category',
       'Property Price to Income Value', 'Property Price to Income Category',
       'Traffic Commute Time Value', 'Traffic Commute Time Category',
       'Pollution Value', 'Pollution Category', 'Quality of Life Value',
       'Quality of Life Category'],
      dtype='object')

In [49]:
qol_1.columns = (
    qol_1.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

qol_1.columns

Index(['country', 'purchasing_power_value', 'purchasing_power_category',
       'safety_value', 'safety_category', 'health_care_value',
       'health_care_category', 'climate_value', 'climate_category',
       'cost_of_living_value', 'cost_of_living_category',
       'property_price_to_income_value', 'property_price_to_income_category',
       'traffic_commute_time_value', 'traffic_commute_time_category',
       'pollution_value', 'pollution_category', 'quality_of_life_value',
       'quality_of_life_category'],
      dtype='object')

In [50]:
qol_1['country'] = qol_1['country'].astype(str).str.strip()
qol_1['country_norm'] = qol_1['country'].str.lower()

In [52]:
#ensure numeric
value_cols = [
    'purchasing_power_value',
    'safety_value',
    'health_care_value',
    'climate_value',
    'cost_of_living_value',
    'property_price_to_income_value',
    'traffic_commute_time_value',
    'pollution_value',
    'quality_of_life_value',
]

for c in value_cols:
    qol_1[c] = pd.to_numeric(qol_1[c], errors='coerce')

In [53]:
# Duplicated country entries
dupes_qol = qol_1[qol_1['country_norm'].duplicated(keep=False)]\
             .sort_values('country_norm')
print(dupes_qol.head(20))

# Missing per column
print(qol_1[value_cols].isna().sum())
print(qol_1[value_cols].isna().mean())

Empty DataFrame
Columns: [country, purchasing_power_value, purchasing_power_category, safety_value, safety_category, health_care_value, health_care_category, climate_value, climate_category, cost_of_living_value, cost_of_living_category, property_price_to_income_value, property_price_to_income_category, traffic_commute_time_value, traffic_commute_time_category, pollution_value, pollution_category, quality_of_life_value, quality_of_life_category, country_norm]
Index: []
purchasing_power_value              0
safety_value                        0
health_care_value                   0
climate_value                       0
cost_of_living_value                0
property_price_to_income_value      3
traffic_commute_time_value          0
pollution_value                     0
quality_of_life_value             114
dtype: int64
purchasing_power_value            0.000000
safety_value                      0.000000
health_care_value                 0.000000
climate_value                     0.000000

In [54]:
for c in value_cols:
    median_val = qol_1[c].median()
    qol_1[c] = qol_1[c].fillna(median_val)

print(qol_1[value_cols].isna().sum())

purchasing_power_value            0
safety_value                      0
health_care_value                 0
climate_value                     0
cost_of_living_value              0
property_price_to_income_value    0
traffic_commute_time_value        0
pollution_value                   0
quality_of_life_value             0
dtype: int64


In [55]:
# Missing values across all columns
print(qol_1.isna().sum())

# Fraction missing
print(qol_1.isna().mean())

country                              0
purchasing_power_value               0
purchasing_power_category            0
safety_value                         0
safety_category                      0
health_care_value                    0
health_care_category                 0
climate_value                        0
climate_category                     0
cost_of_living_value                 0
cost_of_living_category              0
property_price_to_income_value       0
property_price_to_income_category    0
traffic_commute_time_value           0
traffic_commute_time_category        0
pollution_value                      0
pollution_category                   0
quality_of_life_value                0
quality_of_life_category             0
country_norm                         0
dtype: int64
country                              0.0
purchasing_power_value               0.0
purchasing_power_category            0.0
safety_value                         0.0
safety_category                      0.0
he

# deal with "none" and "0.0" and other problem values or sort out use or fill method before saving as clean

In [ ]:
out_path_qol = "../data/country_safety/country_qol_clean.csv"
qol_1.to_csv(out_path_qol, index=False)